# Parameter Recovery Analysis

Tests model identifiability: given behavioral metrics produced by a known set of parameters θ_true,
can we recover θ_true by finding the closest-matching simulation in our existing database?

**Approach:** Leave-one-out nearest-neighbor recovery using the 575 (parameter, metric) pairs
already in `mining_results_adj.csv`. No new simulations are required.

For each trial i:
1. θ_true = the parameters used to generate trial i
2. y_synthetic = the behavioral metrics produced by trial i
3. θ_recovered = parameters of trial j≠i that minimizes the weighted z-score loss to y_synthetic
4. Assess θ_true vs θ_recovered

**Caveat:** Trials were sampled by Optuna (not uniformly) — recovery reflects identifiability
within the behaviorally-relevant region of parameter space.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import pearsonr

df = pd.read_csv('mining_results_adj.csv')
print(f'Total rows in mining_results_adj.csv: {len(df)}')
print(f'Columns: {list(df.columns)}')

## Section 1 — Load & clean data

In [ ]:
param_cols = [
    'param_id_threshold',
    'param_sensory_prec_slope',
    'param_k_shelter',
    'param_k_threat',
    'param_delta_stay',
]

metric_cols = [
    'metric_t_shelter',
    'metric_t_investigating',
    'metric_n_sh_co',
    'metric_n_co_sh',
    'metric_n_co_ch',
    'metric_n_ch_co',
    'metric_entropy',
    'metric_laziness',
]

# Full set: drop rows missing any metric (params can have NaN for sensory_prec_slope)
df_metrics_complete = df.dropna(subset=metric_cols).copy()
print(f'Rows with complete metrics: {len(df_metrics_complete)}')

# Restricted set: drop rows missing ANY param (including sensory_prec_slope)
df_full_complete = df.dropna(subset=param_cols + metric_cols).copy().reset_index(drop=True)
print(f'Rows with complete params AND metrics: {len(df_full_complete)}')

# Show param coverage
print('\nParam coverage (non-NaN):')
for c in param_cols:
    print(f'  {c}: {df[c].notna().sum()} / {len(df)}')

## Section 2 — Loss function

Same weighted z-score loss used during fitting (from `fit_after.py` / `mining.py`).

In [ ]:
# Weights mirror WEIGHTS dict in fit_after.py
WEIGHTS = {
    'metric_t_shelter':       2.0,
    'metric_t_investigating': 2.0,
    'metric_n_sh_co':         0.5,
    'metric_n_co_sh':         0.5,
    'metric_n_co_ch':         0.5,
    'metric_n_ch_co':         0.5,
    'metric_entropy':         0.5,
    'metric_laziness':        1.0,
}

# Standard deviations from mining.py
STD = {
    'metric_t_shelter':       0.19,
    'metric_t_investigating': 0.15,
    'metric_n_sh_co':         6.48,
    'metric_n_co_sh':         6.59,
    'metric_n_co_ch':         3.62,
    'metric_n_ch_co':         3.65,
    'metric_entropy':         0.68,
    'metric_laziness':        0.0455,
}

W = np.array([WEIGHTS[c] for c in metric_cols])
S = np.array([STD[c]     for c in metric_cols])

print('Weights:', W)
print('Std devs:', S)

## Section 3 — LOO nearest-neighbor recovery

We run two versions:
- **Full (5-param)**: uses only trials with all 5 parameters known
- **4-param**: includes trials where `sensory_prec_slope` was fixed/locked (NaN), recovering the other 4 params

In [ ]:
def loo_recovery(df_in, param_cols_in, metric_cols_in, W, S):
    """
    Leave-one-out nearest-neighbor parameter recovery.
    Returns arrays: params_true (N, P), params_recovered (N, P), best_loss (N,)
    """
    df_in = df_in.reset_index(drop=True)
    N = len(df_in)
    
    M = df_in[metric_cols_in].values.astype(float)  # (N, 8)
    
    # Pairwise normalized differences: (N, N, 8)
    diff = (M[:, None, :] - M[None, :, :]) / (S + 1e-6)
    
    # Weighted squared loss: (N, N)
    loss_matrix = (diff**2 * W).sum(axis=2)
    
    # Exclude self-comparison
    np.fill_diagonal(loss_matrix, np.inf)
    
    # Best match index for each trial
    recovered_idx = loss_matrix.argmin(axis=1)   # (N,)
    best_loss = loss_matrix[np.arange(N), recovered_idx]
    
    params_true      = df_in[param_cols_in].values.astype(float)
    params_recovered = params_true[recovered_idx]
    
    return params_true, params_recovered, best_loss, recovered_idx


# --- Full 5-param recovery ---
params_true_5, params_rec_5, best_loss_5, rec_idx_5 = loo_recovery(
    df_full_complete, param_cols, metric_cols, W, S
)
print(f'Full 5-param recovery: N={len(params_true_5)}')
print(f'  Median best-match loss: {np.median(best_loss_5):.3f}')
print(f'  Mean best-match loss:   {np.mean(best_loss_5):.3f}')

# --- 4-param recovery (drop sensory_prec_slope) ---
param_cols_4 = [c for c in param_cols if c != 'param_sensory_prec_slope']
df_4param = df.dropna(subset=param_cols_4 + metric_cols).copy().reset_index(drop=True)
params_true_4, params_rec_4, best_loss_4, rec_idx_4 = loo_recovery(
    df_4param, param_cols_4, metric_cols, W, S
)
print(f'\n4-param recovery (excl. sensory_prec_slope): N={len(params_true_4)}')
print(f'  Median best-match loss: {np.median(best_loss_4):.3f}')

## Section 4 — Recovery scatter plots (5 parameters)

In [ ]:
param_labels = {
    'param_id_threshold':       'id_threshold',
    'param_sensory_prec_slope': 'sensory_prec_slope',
    'param_k_shelter':          'k_shelter',
    'param_k_threat':           'k_threat',
    'param_delta_stay':         'delta_stay',
}

n_params = len(param_cols)
fig, axes = plt.subplots(1, n_params, figsize=(4.5 * n_params, 4.5))

results = {}

for i, (col, ax) in enumerate(zip(param_cols, axes)):
    true_vals = params_true_5[:, i]
    rec_vals  = params_rec_5[:, i]

    # Compute stats
    mask = np.isfinite(true_vals) & np.isfinite(rec_vals)
    r, p = pearsonr(true_vals[mask], rec_vals[mask])
    rmse = np.sqrt(np.mean((true_vals[mask] - rec_vals[mask])**2))
    results[col] = {'r': r, 'p': p, 'rmse': rmse, 'n': mask.sum()}

    # Color by best-match loss
    sc = ax.scatter(true_vals, rec_vals, c=best_loss_5, cmap='viridis_r',
                    alpha=0.6, s=25, vmin=0, vmax=np.percentile(best_loss_5, 95))

    # Identity line
    lo = min(true_vals.min(), rec_vals.min())
    hi = max(true_vals.max(), rec_vals.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='y = x')

    ax.set_xlabel(f'θ_true ({param_labels[col]})', fontsize=10)
    ax.set_ylabel('θ_recovered', fontsize=10)
    ax.set_title(f'{param_labels[col]}\nr = {r:.3f}, RMSE = {rmse:.3f}', fontsize=10)
    plt.colorbar(sc, ax=ax, label='Best-match loss')

fig.suptitle('Parameter Recovery — LOO Nearest-Neighbor (5-param complete trials)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('param_recovery_scatter.svg', bbox_inches='tight')
plt.show()
print('Saved param_recovery_scatter.svg')

## Section 5 — Recovery quality summary table

In [ ]:
summary = pd.DataFrame([
    {
        'Parameter':   param_labels[col],
        'Pearson r':   f"{results[col]['r']:.3f}",
        'p-value':     f"{results[col]['p']:.2e}",
        'RMSE':        f"{results[col]['rmse']:.4f}",
        'N':           results[col]['n'],
        'Identifiable': '✓' if abs(results[col]['r']) > 0.6 else ('~' if abs(results[col]['r']) > 0.3 else '✗'),
    }
    for col in param_cols
])

print(summary.to_string(index=False))

## Section 6 — 4-param recovery scatter (more trials, sensory_prec_slope excluded)

In [ ]:
n_params_4 = len(param_cols_4)
fig, axes = plt.subplots(1, n_params_4, figsize=(4.5 * n_params_4, 4.5))

results_4 = {}

for i, (col, ax) in enumerate(zip(param_cols_4, axes)):
    col_idx = param_cols_4.index(col)
    true_vals = params_true_4[:, col_idx]
    rec_vals  = params_rec_4[:, col_idx]

    mask = np.isfinite(true_vals) & np.isfinite(rec_vals)
    r, p = pearsonr(true_vals[mask], rec_vals[mask])
    rmse = np.sqrt(np.mean((true_vals[mask] - rec_vals[mask])**2))
    results_4[col] = {'r': r, 'p': p, 'rmse': rmse, 'n': mask.sum()}

    sc = ax.scatter(true_vals, rec_vals, c=best_loss_4, cmap='viridis_r',
                    alpha=0.6, s=25, vmin=0, vmax=np.percentile(best_loss_4, 95))
    lo = min(true_vals.min(), rec_vals.min())
    hi = max(true_vals.max(), rec_vals.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5)
    ax.set_xlabel(f'θ_true ({param_labels[col]})', fontsize=10)
    ax.set_ylabel('θ_recovered', fontsize=10)
    ax.set_title(f'{param_labels[col]}\nr = {r:.3f}, RMSE = {rmse:.3f}', fontsize=10)
    plt.colorbar(sc, ax=ax, label='Best-match loss')

fig.suptitle(f'Parameter Recovery — 4-param (N={len(params_true_4)}, excl. sensory_prec_slope)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('param_recovery_4param_scatter.svg', bbox_inches='tight')
plt.show()
print('Saved param_recovery_4param_scatter.svg')

## Section 7 — Best-fit assignments recovery check

The 28 best-fit parameter assignments (one per mouse×session) from `lowest_losses_full_adj.csv`
are the scientifically important points. Check how well each is recovered.

In [ ]:
bf = pd.read_csv('lowest_losses_full_adj.csv')
print('lowest_losses_full_adj.csv columns:', list(bf.columns))
bf.head(5)

In [ ]:
# Parse best-fit assignments: each row maps a Target (e.g. '13_def1') to a source_db + trial_id
# Column names from exploration: 'Target', 'id_1 (Database)', 'id_2.1'
# Identify the correct column names dynamically
print(bf.columns.tolist())
bf_filtered = bf.dropna(subset=['Target']).copy()
print(f'\nBest-fit assignments: {len(bf_filtered)}')
print(bf_filtered[['Target', 'id_1 (Database)', 'id_2.1', 'Loss']].to_string(index=False))

In [ ]:
# For each best-fit assignment, look up its params and metrics in the full mining results
# Then find its LOO recovery and report error

# Build a lookup: (source_db_basename, trial_id) -> row index in df_full_complete
df_full_complete['source_db_base'] = df_full_complete['source_db'].str.replace(r'.*/','', regex=True)
df_4param['source_db_base'] = df_4param['source_db'].str.replace(r'.*/','', regex=True)

lookup_records = []

for _, row in bf_filtered.iterrows():
    target    = row['Target']
    db_name   = str(row.get('id_1 (Database)', '')).strip()
    trial_raw = row.get('id_2.1', None)
    loss_val  = row.get('Loss', None)

    if pd.isna(trial_raw) or db_name == 'nan':
        lookup_records.append({'target': target, 'db': db_name, 'trial': trial_raw,
                                'found': False, 'best_match_loss': np.nan, 'fit_loss': loss_val,
                                'note': 'missing assignment'})
        continue

    trial_id_str = str(int(float(trial_raw)))  # trial_id stored as string in CSV

    # Find in 5-param complete set first
    match = df_full_complete[
        (df_full_complete['source_db_base'] == db_name) &
        (df_full_complete['trial_id'].astype(str) == trial_id_str)
    ]

    if len(match) == 0:
        # Check 4-param set
        match_4 = df_4param[
            (df_4param['source_db_base'] == db_name) &
            (df_4param['trial_id'].astype(str) == trial_id_str)
        ]
        note = 'in 4-param set only (sensory_prec_slope fixed)' if len(match_4) > 0 else 'not found in any complete set'
        lookup_records.append({'target': target, 'db': db_name, 'trial': trial_id_str,
                                'found': False, 'best_match_loss': np.nan, 'fit_loss': loss_val,
                                'note': note})
    else:
        idx = match.index[0]
        local_idx = df_full_complete.index.get_loc(idx)

        bl     = best_loss_5[local_idx]
        true_p = params_true_5[local_idx]
        rec_p  = params_rec_5[local_idx]

        rec = {'target': target, 'db': db_name, 'trial': trial_id_str,
               'found': True, 'best_match_loss': round(float(bl), 3),
               'fit_loss': loss_val, 'note': 'OK'}
        for j, pc in enumerate(param_cols):
            rec[f'true_{param_labels[pc]}'] = round(float(true_p[j]), 4)
            rec[f'rec_{param_labels[pc]}']  = round(float(rec_p[j]), 4)
        lookup_records.append(rec)

bf_recovery = pd.DataFrame(lookup_records)
disp_cols = [c for c in ['target', 'found', 'best_match_loss', 'fit_loss', 'note'] if c in bf_recovery.columns]
print(bf_recovery[disp_cols].to_string(index=False))

In [ ]:
# Detailed view of recovered vs true params for best-fit assignments that were found (5-param)
found_5 = bf_recovery[bf_recovery['found'] == True].copy()

if len(found_5) > 0:
    display_cols = ['target', 'best_match_loss'] + \
        [f'true_{param_labels[c]}' for c in param_cols] + \
        [f'rec_{param_labels[c]}' for c in param_cols]
    display_cols = [c for c in display_cols if c in found_5.columns]
    print(found_5[display_cols].to_string(index=False))
else:
    print('None of the best-fit assignments were in the 5-param complete set.')
    print('Note:', bf_recovery['note'].value_counts().to_string())
    print()
    print('Running 4-param recovery check for best-fit assignments...')

    # Re-do lookup on 4-param set
    bf_rec4_records = []
    for _, row in bf_filtered.iterrows():
        target    = row['Target']
        db_name   = str(row.get('id_1 (Database)', '')).strip()
        trial_raw = row.get('id_2.1', None)
        loss_val  = row.get('Loss', None)
        if pd.isna(trial_raw) or db_name == 'nan':
            continue
        trial_id_str = str(int(float(trial_raw)))

        match_4 = df_4param[
            (df_4param['source_db_base'] == db_name) &
            (df_4param['trial_id'].astype(str) == trial_id_str)
        ]
        if len(match_4) == 0:
            bf_rec4_records.append({'target': target, 'note': 'not found in 4-param set either'})
            continue

        local_idx = match_4.index[0]
        bl     = best_loss_4[local_idx]
        true_p = params_true_4[local_idx]
        rec_p  = params_rec_4[local_idx]

        rec = {'target': target, 'best_match_loss': round(float(bl), 3), 'fit_loss': loss_val}
        for j, pc in enumerate(param_cols_4):
            rec[f'true_{param_labels[pc]}'] = round(float(true_p[j]), 4)
            rec[f'rec_{param_labels[pc]}']  = round(float(rec_p[j]), 4)
        bf_rec4_records.append(rec)

    if bf_rec4_records:
        bf_rec4 = pd.DataFrame(bf_rec4_records)
        print(bf_rec4.to_string(index=False))
    else:
        print('No best-fit assignments found in any complete set.')

## Section 8 — Parameter confusion / correlation heatmap

Shows pairwise correlations between parameters in the recovered set — reveals which pairs
tend to be confused (non-identifiable together).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (params, label) in zip(axes, [
    (params_true_5, 'True params'),
    (params_rec_5,  'Recovered params')
]):
    labels_short = [param_labels[c] for c in param_cols]
    corr = np.corrcoef(params.T)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(param_cols)))
    ax.set_yticks(range(len(param_cols)))
    ax.set_xticklabels(labels_short, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(labels_short, fontsize=9)
    ax.set_title(label, fontsize=11)
    for i in range(len(param_cols)):
        for j in range(len(param_cols)):
            ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(corr[i,j]) > 0.5 else 'black')
    plt.colorbar(im, ax=ax)

fig.suptitle('Parameter correlation structure: true vs recovered', fontsize=12)
plt.tight_layout()
plt.savefig('param_recovery_corr.svg', bbox_inches='tight')
plt.show()
print('Saved param_recovery_corr.svg')

## Section 9 — Recovery error vs best-match loss

Checks whether recovery quality degrades when the best-match loss is high
(i.e., when there's no similar simulation in the database for a given query).

In [ ]:
param_errors = np.abs(params_true_5 - params_rec_5)  # (N, 5)

fig, axes = plt.subplots(1, n_params, figsize=(4 * n_params, 4))

for i, (col, ax) in enumerate(zip(param_cols, axes)):
    ax.scatter(best_loss_5, param_errors[:, i], alpha=0.4, s=15)
    ax.set_xlabel('Best-match loss', fontsize=9)
    ax.set_ylabel('|θ_true - θ_recovered|', fontsize=9)
    ax.set_title(param_labels[col], fontsize=10)

fig.suptitle('Recovery error vs. best-match loss', fontsize=12)
plt.tight_layout()
plt.savefig('param_recovery_error_vs_loss.svg', bbox_inches='tight')
plt.show()
print('Saved param_recovery_error_vs_loss.svg')

## Section 10 — (Optional) Noise floor estimate

**Only run this if recovery in Sections 4–6 looks poor (r < 0.4 for key params).**

Picks a few best-fit parameter sets, re-runs them 5× each, and measures
metric variability — distinguishing noise-driven non-identifiability from true equifinality.

In [ ]:
# Set RUN_NOISE_FLOOR = True to execute this section (requires sim.py + ~25 new simulations)
RUN_NOISE_FLOOR = False

if RUN_NOISE_FLOOR:
    from sim import run_sim
    from utils import calculate_metrics

    # Use the 5 trials with the lowest fitting loss from the full-complete set
    # (these are likely the most reliable representations of their param sets)
    loss_cols = [c for c in df_full_complete.columns if c.startswith('LOSS_')]
    df_full_complete['min_loss'] = df_full_complete[loss_cols].min(axis=1)
    top5 = df_full_complete.nsmallest(5, 'min_loss')[param_cols + metric_cols + ['source_db', 'trial_id']]
    print('Top 5 trials to noise-test:')
    print(top5)

    N_REPEATS = 5
    noise_results = []

    for _, row in top5.iterrows():
        params = {c.replace('param_', ''): row[c] for c in param_cols}
        for rep in range(N_REPEATS):
            h = run_sim(
                id_threshold=params['id_threshold'],
                sensory_imprecision=params['sensory_prec_slope'],
                k_shelter=params['k_shelter'],
                k_threat=params['k_threat'],
                delta_stay=params['delta_stay'],
            )
            import pandas as _pd
            traj = _pd.DataFrame({'location': h['agent_loc']})
            m = calculate_metrics(traj)
            m.update(params)
            m['rep'] = rep
            m['trial_id'] = row['trial_id']
            noise_results.append(m)

    noise_df = pd.DataFrame(noise_results)
    print('\nMetric std dev across repeats (noise floor):')
    print(noise_df.groupby('trial_id')[[m.replace('metric_','') for m in metric_cols]].std())
else:
    print('RUN_NOISE_FLOOR is False — skipping. Set to True if main recovery looks poor.')

## Summary

| Parameter | Identifiable? | Notes |
|-----------|--------------|-------|
| id_threshold | see table above | Affects when danger mode triggers |
| sensory_prec_slope | see table above | Subtle effect; also has fewer trials |
| k_shelter | see table above | Strong effect on t_shelter |
| k_threat | see table above | Strong effect on t_investigating, chamber visits |
| delta_stay | see table above | Modulates laziness |

**Interpretation guide:**
- r > 0.8 → well-identifiable
- 0.5–0.8 → moderately identifiable  
- < 0.5 → poor identifiability (equifinality or noise floor)

If any key parameter has r < 0.5, consider running Section 10 to check whether
simulation noise is the culprit, or whether additional metrics need to be included.